# T20 Score Predictor — Gradio Interface

Interactive UI for predicting the T20 first-innings final score.

Enter the raw match state — the app calculates **Current Run Rate** and
**Remaining Batting Strength Score** automatically using the same weights as training,
then calls the fine-tuned LLaMA 3.2-3B model via the deployed Modal service (`t20-scorer-service`).

Make sure you have run `modal deploy -m t20_scorer_service` before using this notebook.

In [1]:
%pip install gradio --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import modal
import gradio as gr
from dotenv import load_dotenv

load_dotenv(override=True)

# Connect to the deployed Modal service
Scorer = modal.Cls.from_name("t20-scorer-service", "Scorer")
scorer = Scorer()

/Users/damith/projects/t20_score_prediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import json

# Weights used during training (from 05_data_processing_remaining_depth.ipynb)
BATTING_WEIGHTS = {
    "batter": 1.0,
    "all_rounder": 0.8,
    "utility_player": 0.65,
    "bowler": 0.3,
    "unknown": 0.45,   # INSUFFICIENT_DATA default
}


def calculate_batting_strength(
    remaining_batters: int,
    remaining_all_rounders: int,
    remaining_utility_players: int,
    remaining_bowlers: int,
    remaining_unknown: int,
) -> float:
    """Replicate calculate_batting_strength_score() from notebook 05."""
    score = (
        remaining_batters           * BATTING_WEIGHTS["batter"]
        + remaining_all_rounders    * BATTING_WEIGHTS["all_rounder"]
        + remaining_utility_players * BATTING_WEIGHTS["utility_player"]
        + remaining_bowlers         * BATTING_WEIGHTS["bowler"]
        + remaining_unknown         * BATTING_WEIGHTS["unknown"]
    )
    return round(score, 2)


def build_prompt(
    venue, par_score, batting_team,
    overs_completed, balls_remaining,
    runs_scored, wickets_lost,
    runs_last_2_overs, dot_balls_total, dot_balls_last_2,
    fours, sixes, powerplay_completed,
    remaining_batters, remaining_all_rounders,
    remaining_utility_players, remaining_bowlers, remaining_unknown,
) -> tuple[str, float, float]:
    """Build the model prompt from raw inputs; also returns computed run_rate and batting_strength."""
    run_rate = round(runs_scored / overs_completed, 2) if overs_completed > 0 else 0.0
    batting_strength = calculate_batting_strength(
        remaining_batters, remaining_all_rounders,
        remaining_utility_players, remaining_bowlers, remaining_unknown,
    )
    powerplay_str = "Yes" if powerplay_completed else "No"
    prompt = (
        f"Match type: T20\n"
        f"Venue: {venue}\n"
        f"Venue par score: {par_score}\n"
        f"Batting team: {batting_team}\n"
        f"\n"
        f"Overs completed: {overs_completed}\n"
        f"Balls remaining: {int(balls_remaining)}\n"
        f"Runs scored: {int(runs_scored)}\n"
        f"Wickets lost: {int(wickets_lost)}\n"
        f"Current run rate: {run_rate}\n"
        f"\n"
        f"Runs in last 2 overs: {int(runs_last_2_overs)}\n"
        f"Dot balls so far: {int(dot_balls_total)}\n"
        f"Dot balls in last 2 overs: {int(dot_balls_last_2)}\n"
        f"\n"
        f"Fours hit: {int(fours)}\n"
        f"Sixes hit: {int(sixes)}\n"
        f"\n"
        f"Powerplay completed: {powerplay_str}\n"
        f"\n"
        f"Remaining Batting Strength Score: {batting_strength}\n"
        f"\n"
        f"Final 1st innings score:"
    )
    return prompt, run_rate, batting_strength


def predict_score(
    venue, par_score, batting_team,
    overs_completed, balls_remaining,
    runs_scored, wickets_lost,
    runs_last_2_overs, dot_balls_total, dot_balls_last_2,
    fours, sixes, powerplay_completed,
    remaining_batters, remaining_all_rounders,
    remaining_utility_players, remaining_bowlers, remaining_unknown,
):
    prompt, run_rate, batting_strength = build_prompt(
        venue, par_score, batting_team,
        overs_completed, balls_remaining,
        runs_scored, wickets_lost,
        runs_last_2_overs, dot_balls_total, dot_balls_last_2,
        fours, sixes, powerplay_completed,
        remaining_batters, remaining_all_rounders,
        remaining_utility_players, remaining_bowlers, remaining_unknown,
    )
    info = f"Run rate: {run_rate:.2f}  |  Batting Strength Score: {batting_strength}"
    try:
        score = scorer.predict.remote(prompt)
        return f"**Predicted Final Score: {score:.0f} runs**", info, prompt
    except Exception as e:
        return f"Error: {e}", info, prompt


# Expected JSON keys and their defaults (mirrors the form fields)
JSON_DEFAULTS = {
    "venue": "Melbourne Cricket Ground",
    "par_score": 165,
    "batting_team": "Australia",
    "overs_completed": 10,
    "balls_remaining": 60,
    "runs_scored": 82,
    "wickets_lost": 2,
    "runs_last_2_overs": 18,
    "dot_balls_total": 22,
    "dot_balls_last_2": 3,
    "fours": 8,
    "sixes": 3,
    "powerplay_completed": True,
    "remaining_batters": 3,
    "remaining_all_rounders": 3,
    "remaining_utility": 1,
    "remaining_bowlers": 3,
    "remaining_unknown": 0,
}


def load_from_json(file):
    """Parse an uploaded JSON file and return values for all form fields."""
    if file is None:
        return [JSON_DEFAULTS[k] for k in JSON_DEFAULTS]
    try:
        with open(file, "r") as f:
            data = json.load(f)
        return [data.get(k, JSON_DEFAULTS[k]) for k in JSON_DEFAULTS]
    except Exception as e:
        raise gr.Error(f"Could not parse JSON file: {e}")


In [10]:
with gr.Blocks(title="T20 Score Predictor") as demo:
    gr.Markdown("# T20 First-Innings Score Predictor")
    gr.Markdown(
        "Enter the raw match state, or **upload a JSON file** to populate all fields at once. "
        "The app calculates **Current Run Rate** and **Remaining Batting Strength Score** "
        "automatically before calling the model."
    )

    # ── JSON upload ──────────────────────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=2):
            json_upload = gr.File(
                label="Upload match state JSON (optional)",
                file_types=[".json"],
                file_count="single",
            )
        with gr.Column(scale=3):
            gr.Markdown(
                "**Expected JSON keys:**\n"
                "`venue`, `par_score`, `batting_team`, `overs_completed`, `balls_remaining`, "
                "`runs_scored`, `wickets_lost`, `runs_last_2_overs`, `dot_balls_total`, "
                "`dot_balls_last_2`, `fours`, `sixes`, `powerplay_completed` *(bool)*, "
                "`remaining_batters`, `remaining_all_rounders`, `remaining_utility`, "
                "`remaining_bowlers`, `remaining_unknown`"
            )

    gr.Markdown("---")

    # ── Manual inputs ────────────────────────────────────────────────────────
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Venue & Teams")
            venue        = gr.Textbox(label="Venue")
            par_score    = gr.Number(label="Venue Par Score", precision=0)
            batting_team = gr.Textbox(label="Batting Team")

        with gr.Column():
            gr.Markdown("### Innings State")
            overs_completed = gr.Slider(label="Overs Completed", minimum=0, maximum=20, step=1)
            balls_remaining = gr.Number(label="Balls Remaining", precision=0)
            runs_scored     = gr.Number(label="Runs Scored", precision=0)
            wickets_lost    = gr.Slider(label="Wickets Lost", minimum=0, maximum=10, step=1)

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Recent Play")
            runs_last_2_overs   = gr.Number(label="Runs in Last 2 Overs", precision=0)
            dot_balls_total     = gr.Number(label="Dot Balls So Far", precision=0)
            dot_balls_last_2    = gr.Number(label="Dot Balls in Last 2 Overs", precision=0)
            fours               = gr.Number(label="Fours Hit", precision=0)
            sixes               = gr.Number(label="Sixes Hit", precision=0)
            powerplay_completed = gr.Checkbox(label="Powerplay Completed")

        with gr.Column():
            gr.Markdown("### Remaining Batting Lineup")
            gr.Markdown(
                "Count players **not yet dismissed** by role. "
                "The Batting Strength Score is calculated using the same weights as training: "
                "Batter=1.0 · All-rounder=0.8 · Utility=0.65 · Bowler=0.3 · Unknown=0.45"
            )
            remaining_batters      = gr.Slider(label="Specialist Batters remaining", minimum=0, maximum=11, step=1)
            remaining_all_rounders = gr.Slider(label="All-rounders remaining",       minimum=0, maximum=11, step=1)
            remaining_utility      = gr.Slider(label="Utility players remaining",    minimum=0, maximum=11, step=1)
            remaining_bowlers      = gr.Slider(label="Bowlers remaining",            minimum=0, maximum=11, step=1)
            remaining_unknown      = gr.Slider(label="Unknown / insufficient data",  minimum=0, maximum=11, step=1)

    predict_btn = gr.Button("Predict Score", variant="primary")

    score_output = gr.Markdown(label="Prediction")

    with gr.Accordion("Calculated values", open=False):
        calc_output = gr.Textbox(label="Calculated values", interactive=False, show_label=False)

    with gr.Accordion("Prompt sent to model", open=False):
        prompt_output = gr.Textbox(label="Prompt sent to model", lines=22, interactive=False, show_label=False)

    # All form fields in the same order as JSON_DEFAULTS keys
    all_inputs = [
        venue, par_score, batting_team,
        overs_completed, balls_remaining,
        runs_scored, wickets_lost,
        runs_last_2_overs, dot_balls_total, dot_balls_last_2,
        fours, sixes, powerplay_completed,
        remaining_batters, remaining_all_rounders,
        remaining_utility, remaining_bowlers, remaining_unknown,
    ]

    # Wire JSON upload → populate all fields
    json_upload.upload(
        fn=load_from_json,
        inputs=[json_upload],
        outputs=all_inputs,
    )

    # Wire predict button
    predict_btn.click(
        fn=predict_score,
        inputs=all_inputs,
        outputs=[score_output, calc_output, prompt_output],
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
